## <font color="greenblue">Processamento - uma única tabela</font>

In [1]:
import pandas as pd
import numpy as np
# from statsmodels.stats.proportion import proportions_ztest
from collections import Counter
from utils import ordenar_labels, ordenar_labels_multipla, ordenar_valores, classificar_nps, funcao_agrupamento, classificar_satis

TipoTabela = 'SIMPLES'
Colunas = 'MERC_PME, MERC_500, TIM_TOT, TIM_PME, TIM_500, TIM_PRIME, VIV_TOT, VIV_PME, VIV_500, CLA_TOT, CLA_PME, CLA_500'
Cabecalho = 'Mercado PME, Mercado 500+, TIM, TIM PME, TIM 500+, TIM Prime, VIVO, VIVO PME, VIVO 500+, CLARO, CLARO PME, CLARO 500+'
Var_linha = 'Q2' 
NS_NR = 'NAO'
valores_BTB = '1, 2'
valores_TTB = '4, 5'
Valores_Agrup = 'Com certeza deixaria de trabalhar, Provavelmente deixaria de trabalhar, Não sei se vou continuar ou não, Provavelmente continuaria trabalhando, Com certeza continuaria trabalhando'
Fecha_100 = 'NAO'
Var_ID = 'codigo_entrevistado' # codigo_entrevistado | ID_EMP
Var_Pond = 'POND'
Titulo = '2. COTA - Qual o número de funcionários registrados da empresa?  (Espontânea - RU)'

In [2]:
path = r'C:\Users\rayner.santos\Downloads\Consistencia e Processamento\BD_TIM_Imagem_Onda1_2025_v10.xlsx'
data = pd.read_excel(path, sheet_name='BD_CODIGOS')
df = data.copy()
df

,codigo_entrevistado,F_1,F_2,F3,F4,Q1,Q2,Q2_FX,Q2A,Q2A_OU,...,SEG_VIV3,SEG_VIV4,SEG_VIV5,SEG_CLA1,SEG_CLA2,SEG_CLA3,SEG_CLA4,SEG_CLA5,PME_500,xxx
0,10011,2,2,2,1,5,1,3,2,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1,NaN
1,10058,2,2,2,1,4,6,100,2,NaN,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,5.0,1,NaN
2,10073,2,2,2,1,9,1,3,2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,1,NaN
3,10125,2,2,2,1,3,1,1,2,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,1,NaN
4,10206,2,2,2,1,1,1,3,2,NaN,...,3.0,NaN,NaN,NaN,2.0,NaN,NaN,NaN,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3039,390846,2,2,2,1,8,6,100,2,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1,NaN
3040,390847,2,2,2,1,10,6,180,2,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1,NaN
3041,390850,2,2,2,1,8,6,140,2,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,1,NaN
3042,390851,2,2,2,1,10,6,140,2,NaN,...,NaN,4.0,NaN,NaN,NaN,NaN,NaN,5.0,1,NaN


In [3]:
lista_labels = pd.read_excel(path, sheet_name='Lista de Labels')
lista_labels = lista_labels.iloc[1:, :].copy()
lista_labels.columns = ['Coluna', 'Codigo', 'Label']
lista_labels["Coluna"] = lista_labels["Coluna"].ffill().str.strip()
lista_labels

,Coluna,Codigo,Label
1,F_1,1.0,Sim
2,F_1,2.0,Não
3,F_2,1.0,Sim
4,F_2,2.0,Não
5,F3,1.0,Sim
...,...,...,...
14991,SEG_CLA5,3.0,Vulneráveis
14992,SEG_CLA5,4.0,Rejeitadores
14993,SEG_CLA5,5.0,Leais
14994,PME_500,1.0,PME


In [13]:
# Variáveis para as colunas da tabela (bandeiras)
Colunas = Colunas.split(sep=', ')
print(Colunas)
dict_ord_labels = {}

for col in Colunas:
    if col not in df.columns:
        raise ValueError(f"Coluna '{col}' não encontrada no DataFrame.")        
    else:
        df[col], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=col)
        dict_ord_labels[col] = ord_labels
        print(f"\nColuna ordenada (unique): {df[col].unique()}")

['MERC_PME', 'MERC_500', 'TIM_TOT', 'TIM_PME', 'TIM_500', 'TIM_PRIME', 'VIV_TOT', 'VIV_PME', 'VIV_500', 'CLA_TOT', 'CLA_PME', 'CLA_500']

#== VARIÁVEL SENDO PROCESSADA: MERC_PME ==#
MERC_PME
2.0    2754
Name: count, dtype: int64 

MERC_PME
2.0    1.0
Name: proportion, dtype: float64

Labels filtrados para a coluna alvo:
        Codigo         Label
14779       1       Mercado
14780       2   Mercado PME
14781       3  Mercado 500+

df[Variavel] (antes de ordenar):
0    2
1    2
2    2
3    2
4    2
5    2
6    2
7    2
8    2
9    2
Name: MERC_PME, dtype: Int64

Ordem numérica encontrada: [2]

Ordem mapeada com labels:
    Codigo        Label
0       2  Mercado PME

Ordem final com labels: ['Mercado PME']

Coluna categórica ordenada criada: ['Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME', 'Mercado PME']
Categories (1, object): ['Mercado PME']
MERC_PME
Mercado PME    2754
Name: count, dtype: int64 

M

In [14]:
# Etapas de ETL para as colunas principais a serem utilizadas
# Transformação na variável para a linha da tabela
if TipoTabela == 'SIMPLES':
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
    else:
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
    
elif (TipoTabela == 'NPS') | (TipoTabela == 'IPA_10'):
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha] = df[Var_linha].replace('ns/nr', np.nan)
        df[Var_linha] = df[Var_linha].replace('90', np.nan)
        df[Var_linha] = df[Var_linha].replace('99', np.nan)
        df[Var_linha] = df[Var_linha].replace('999', np.nan)
        df[Var_linha] = df[Var_linha].replace('9999', np.nan)
        df[Var_linha] = df[Var_linha].replace(90, np.nan)
        df[Var_linha] = df[Var_linha].replace(99, np.nan)
        df[Var_linha] = df[Var_linha].replace(999, np.nan)
        df[Var_linha] = df[Var_linha].replace(9999, np.nan)
        df[Var_linha] = pd.to_numeric(df[Var_linha], errors='coerce', downcast='integer')
        # df[Var_linha], ord_labels = ordenar_labels(df=df, lista_labels=lista_labels, Variavel=Var_linha)
        df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

        if TipoTabela == 'NPS':
            df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

        elif TipoTabela == 'IPA_10':
            valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

    else:
        # df[Var_linha] = df[Var_linha].replace('90', np.nan)
        # df[Var_linha] = df[Var_linha].replace('99', np.nan)
        # df[Var_linha] = df[Var_linha].replace('999', np.nan)
        # df[Var_linha] = df[Var_linha].replace('9999', np.nan)
        # df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

        if TipoTabela == 'NPS':
            df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

        elif TipoTabela == 'IPA_10':
            valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

elif TipoTabela == 'IPA_5':
    if NS_NR == 'NAO':
        df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
        df[Var_linha] = df[Var_linha].replace('90', np.nan)
        df[Var_linha] = df[Var_linha].replace('99', np.nan)
        df[Var_linha] = df[Var_linha].replace('999', np.nan)
        df[Var_linha] = df[Var_linha].replace(90, np.nan)
        df[Var_linha] = df[Var_linha].replace(99, np.nan)
        df[Var_linha] = df[Var_linha].replace(999, np.nan)
        valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
        valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
        df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
        df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)
    else:
        valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
        valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
        df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
        df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
        df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
        # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)

elif TipoTabela == 'MULTIPLA':
    if NS_NR == 'NAO':
        df_NS_NR = df.copy()
        
        Valores_Agrup = Valores_Agrup.split(sep=',')
        # Converte as colunas de motivos para string, preservando NaN
        # for c in Valores_Agrup:
        #     df[c] = df[c].astype("object").where(df[c].isna(), df[c].astype(str))
        #     df_NS_NR[c] = df_NS_NR[c].astype("object").where(df_NS_NR[c].isna(), df_NS_NR[c].astype(str))

        bd_motivo = pd.melt(df, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('90', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('99', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('999', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('9999', np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(90, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(99, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(999, np.nan)
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(9999, np.nan)
        bd_motivo = bd_motivo.dropna(subset=[Var_linha])
        print(f'\nbd_motivo em formato de código:\n{bd_motivo}')
        bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
        bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('NS/NR', np.nan)
        # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)  

        df_limpo = bd_motivo.dropna(subset=[Var_linha])
        print(f'bd_motivo finalizado:\n{df_limpo}')
        df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')
                
        # Bancos para realizar o cálculo do Índice de Multiplicidade (incluir NS/NR)
        bd_motivo_NS_NR = pd.melt(df_NS_NR, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo_NS_NR = bd_motivo_NS_NR.dropna(subset=[Var_linha])
        bd_motivo_NS_NR = ordenar_labels_multipla(df=bd_motivo_NS_NR, lista_labels=lista_labels, Variavel=Var_linha, 
                                                  Var_Valores_Agrup=Valores_Agrup[0])
        # bd_motivo_NS_NR[Var_linha] = pd.Categorical(bd_motivo_NS_NR[Var_linha], 
        #                                     categories=ordenar_valores(bd_motivo_NS_NR[Var_linha]), ordered=True)
        
        df_NS_NR_limpo = bd_motivo_NS_NR.dropna(subset=[Var_linha])
        print(f'bd_motivo_NS_NR finalizado:\n{df_limpo}')
        df_NS_NR_unico = df_NS_NR_limpo.drop_duplicates(subset=Var_ID, keep='first')    
        
    else:
        Valores_Agrup = Valores_Agrup.split(sep=',')
        bd_motivo = pd.melt(df, 
                    id_vars=Colunas + [Var_Pond] + [Var_ID],
                    value_vars=Valores_Agrup, 
                    var_name='Valores', 
                    value_name=Var_linha)
        bd_motivo = bd_motivo.dropna(subset=[Var_linha])
        bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
        # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], 
        #                                     categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)
        
        df_limpo = bd_motivo.dropna(subset=[Var_linha])
        df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')


#== VARIÁVEL SENDO PROCESSADA: Q2 ==#
Q2
1    695
6    666
3    635
4    330
2    286
8    188
9    102
7     81
5     61
Name: count, dtype: int64 

Q2
1    0.228318
6    0.218791
3    0.208607
4    0.108410
2    0.093955
8    0.061761
9    0.033509
7    0.026610
5    0.020039
Name: proportion, dtype: float64

Labels filtrados para a coluna alvo:
     Codigo                           Label
19       1              1 a 4 funcionários
20       2                        De 5 a 9
21       3                      De 10 a 19
22       4                      De 20 a 49
23       5         De 50 a 99 funcionários
24       6       De 100 a 249 funcionários
25       7       De 250 a 499 funcionários
26       8       De 500 a 999 funcionários
27       9       Acima de 999 funcionários
28      90  Não quer responder ou não sabe

df[Variavel] (antes de ordenar):
0    1
1    6
2    1
3    1
4    1
5    3
6    1
7    1
8    2
9    3
Name: Q2, dtype: Int64

Ordem numérica encontrada: [1, 2, 3, 4, 5, 6, 7

In [15]:
#======= Criação da Tabela Geral =======#
tabela_geral = []
tabelas_pond = []
tabelas_pond_freq_abs = []
tabelas_sem_pond = []
aux_tabelas_pond = []
aux_tabelas_sem_pond = []

tbl_pond_freq_abs_respondentes = []
tbl_sem_pond_respondentes = []

tbl_pond_freq_abs_respostas_NS_NR = []
tbl_pond_freq_abs_respondentes_NS_NR = []

if TipoTabela == 'MULTIPLA':
    if NS_NR == 'NAO':
        banco = df_NS_NR_unico
    else:
        banco = df_unico

    if NS_NR == 'NAO':
        banco_pivotado = bd_motivo_NS_NR
    else:
        banco_pivotado = bd_motivo

    i = 0    
    for col in Colunas:

        # Gerar Tabelas Ponderadas de Frequência Absoluta com o banco empilhado
        tabela = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        if Fecha_100 == 'SIM':
            tabela = tabela.div(tabela.sum())
            tabelas_pond.append(tabela)
        else:
            tabelas_pond.append(tabela)

        tabela = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respostas_NS_NR.append(tabela)

        tabela = pd.pivot_table(df_unico, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respondentes.append(tabela)

        tabela = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, columns=col, aggfunc='sum')
        tbl_pond_freq_abs_respondentes_NS_NR.append(tabela)

        tabela = pd.crosstab(df_unico[Var_linha], df_unico[col], dropna=False)
        tabela = tabela.fillna(0)
        if len(tabela) == 0:
            tabela = pd.DataFrame(0, index=df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique(), 
                                columns=df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique())
            print(f'{df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique()}\n')
            print(f'{df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique()}\n')
        print(f'tabela:\n{tabela}')
        tbl_sem_pond_respondentes.append(tabela)
        i += 1

    tabela_geral = pd.concat(tabelas_pond, axis=1)
    tbl_pond_freq_abs_respondentes = pd.concat(tbl_pond_freq_abs_respondentes, axis=1)
    tbl_sem_pond_respondentes = pd.concat(tbl_sem_pond_respondentes, axis=1)
    tbl_pond_freq_abs_respostas_NS_NR = pd.concat(tbl_pond_freq_abs_respostas_NS_NR, axis=1)
    tbl_pond_freq_abs_respondentes_NS_NR = pd.concat(tbl_pond_freq_abs_respondentes_NS_NR, axis=1)
    
    if Fecha_100 != 'SIM':
        soma_base_ponderada = tbl_pond_freq_abs_respondentes.sum()
        tabela_geral = tabela_geral.div(soma_base_ponderada)
    
    print(f'{tabela_geral}\n')


    # Trazendo a coluna de valores gerais
    soma_colunas = tbl_pond_freq_abs_respondentes.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    valores_gerais = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, aggfunc='sum')
    if Fecha_100 == 'SIM':
        percentual_geral = valores_gerais.div(valores_gerais.sum()).sort_index()
        print(f'\npercentual_geral:\n{percentual_geral}')
        print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')
    else:
        percentual_geral = valores_gerais.div(base_ponderada[0]).sort_index()
        print(f'\npercentual_geral:\n{percentual_geral}')
        print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')

    # tabela_geral = tabela_geral.div(base_ponderada[1:])
    tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
    print(f'{tabela_geral}\n')

    # Valores para Base Ponderada
    soma_colunas = tbl_pond_freq_abs_respondentes.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    tabela_geral.loc['Base Ponderada'] = base_ponderada

    # Valores para Base Sem Ponderar
    valores_gerais_respondentes = df_unico[Var_ID].value_counts()
    soma_colunas = pd.Series(tbl_sem_pond_respondentes.sum())
    print(f'\n{soma_colunas}\n')
    base_sem_ponderar = pd.Series(valores_gerais_respondentes.sum())
    base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
    print(f'base sem ponderar:\n{base_sem_ponderar.index}\n')
    print(f'{len(base_sem_ponderar.index)}\n')
    print(f'tabela geral:\n{tabela_geral.columns}\n')
    print(f'{len(tabela_geral.columns)}\n')
    base_sem_ponderar.index = tabela_geral.columns
    tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

    # Valores para Base Ponderada - Respostas
    soma_colunas_respostas = tbl_pond_freq_abs_respostas_NS_NR.sum()
    soma_colunas_respostas = pd.Series(soma_colunas_respostas)
    base_ponderada_respostas = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, aggfunc='sum')
    base_ponderada_respostas = pd.Series(base_ponderada_respostas.sum())
    base_ponderada_respostas = pd.concat([base_ponderada_respostas, soma_colunas_respostas])

    # Valores para Base Ponderada - Respondentes
    soma_colunas = tbl_pond_freq_abs_respondentes_NS_NR.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, aggfunc='sum')
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])

    # Índice de Multiplicidade (Total de respostas / Total de respondentes)
    indice_multiplicidade = base_ponderada_respostas / base_ponderada
    tabela_geral.loc['Índice de Multiplicidade'] = indice_multiplicidade

    tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
    print(f'{tabela_geral}\n')

else:
    if df[Var_linha].isna().all():
        # df[Var_linha] = df[Var_linha].astype('float').fillna(0)
        df[Var_linha] = df[Var_linha].cat.add_categories(['Não possui categoria']).fillna('Não possui categoria')
        print(f'Variável em branco\n{df[Var_linha]}\n')

    for col in Colunas:
        # Gerar Tabelas Ponderadas de frequência absoluta
        tabela = pd.crosstab(df[Var_linha], df[col], values=df[Var_Pond], aggfunc='sum')
        tabelas_pond_freq_abs.append(tabela)

        # Gerar Tabelas Ponderadas de frequência relativa 
        tabela = tabela.div(tabela.sum())
        tabela = tabela.fillna(0)
        tabelas_pond.append(tabela)
        print(f'Tabela Ponderada Freq Rel: \n{tabela}\n')

        # Gerar Tabelas Sem Ponderação
        tabela = pd.crosstab(df[Var_linha], df[col], dropna=False)
        tabela = tabela.fillna(0)
        print(f'Tabela Sem Ponderação: \n{tabela}\n')
        if len(tabela) == 0:
            tabela = pd.DataFrame(0, index=df[Var_linha][pd.notna(df[Var_linha])].unique(), 
                                columns=df[col][pd.notna(df[col])].unique())
            print(f'Tabela Sem Ponderação: \n{tabela}\n')
        tabelas_sem_pond.append(tabela)

        # Gerar Tabelas para valores agrupados
        if 'var_agrupada' in df.columns:
            tabela = pd.crosstab(df['var_agrupada'], df[col], values=df[Var_Pond], aggfunc='sum')
            tabela = tabela.div(tabela.sum())
            print(f'Tabela Valores Agrupados: \n{tabela}\n')
            aux_tabelas_pond.append(tabela)

            # Tabelas sem ponderação
            tabela = pd.crosstab(df['var_agrupada'], df[col], dropna=False)
            print(f'Tabela Valores Agrupados Sem Ponderação: \n{tabela}\n')
            aux_tabelas_sem_pond.append(tabela)

    tabela_geral = pd.concat(tabelas_pond, axis=1)
    tabelas_pond_freq_abs = pd.concat(tabelas_pond_freq_abs, axis=1)
    tabelas_sem_pond = pd.concat(tabelas_sem_pond, axis=1)

    # Trazendo a coluna de valores gerais
    valores_gerais_pond = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
    print(f'\nValores GERAL:\n{valores_gerais_pond}')
    percentual_geral = valores_gerais_pond.div(valores_gerais_pond.sum()).sort_index()
    print(f'\PERCENTUAL GERAL:\n{percentual_geral}')

    if 'var_agrupada' in df.columns:
        aux_tabelas_pond = pd.concat(aux_tabelas_pond, axis=1)
        aux_tabelas_sem_pond = pd.concat(aux_tabelas_sem_pond, axis=1)

        # Percentual para os valores gerais da variável agrupada
        valores_gerais_aux = pd.pivot_table(df, values=Var_Pond, index='var_agrupada', aggfunc='sum')
        percentual_geral_aux = valores_gerais_aux.div(valores_gerais_aux.sum()).sort_index()
        tabela_geral = pd.concat([tabela_geral, aux_tabelas_pond], axis=0)
        percentual_geral = pd.concat([percentual_geral, percentual_geral_aux], axis=0)
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
    else:
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)

    # Multiplicando cada linha pelo seu índice+1
    if TipoTabela == 'NPS':
        # Gerar o NPS
        nps = tabela_geral.loc['Promotor'] - tabela_geral.loc['Detrator']
        tabela_geral.loc['NPS'] = nps
        # Gerar as médias
        media_tabela = tabelas_pond_freq_abs.copy()
        for i in range(len(tabelas_pond_freq_abs)):
            media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
        media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

        # Cálculo da média geral
        df_limpo = df.dropna(subset=[Var_linha])
        media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
        media_geral = pd.Series(media_geral)
        media_tabela = pd.concat([media_geral, media_tabela], axis=0)
        media_tabela.index = tabela_geral.columns
        tabela_geral.loc['Media'] = media_tabela

    elif (TipoTabela == 'SATISFACAO') | (TipoTabela == 'IPA_10') :
        media_tabela = tabelas_pond_freq_abs.copy()
        for i in range(len(tabelas_pond_freq_abs)):
            media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
        media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

        # Cálculo da média geral
        df_limpo = df.dropna(subset=[Var_linha])
        media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
        media_geral = pd.Series(media_geral)
        media_tabela = pd.concat([media_geral, media_tabela], axis=0)

        media_tabela.index = tabela_geral.columns
        tabela_geral.loc['Media'] = media_tabela

    else:
        tabela_geral

    tabela_geral = tabela_geral.fillna(0)

    # Valores para Base Ponderada
    tabelas_pond_freq_abs = tabelas_pond_freq_abs.fillna(0)
    soma_colunas = tabelas_pond_freq_abs.sum()
    soma_colunas = pd.Series(soma_colunas)
    base_ponderada = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
    base_ponderada = pd.Series(base_ponderada.sum())
    base_ponderada = pd.concat([base_ponderada, soma_colunas])
    # base_ponderada.index = tabela_geral.columns
    tabela_geral.loc['Base Ponderada'] = base_ponderada

    # Valores para Base Sem Ponderar
    tabelas_sem_pond = tabelas_sem_pond.fillna(0)
    soma_colunas = pd.Series(tabelas_sem_pond.sum())
    print(f'{soma_colunas}\n')
    valores_gerais = df[Var_linha].value_counts().sort_index()
    base_sem_ponderar = pd.Series(valores_gerais.sum())
    base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
    print(f'{base_sem_ponderar.index}\n')
    print(f'{tabela_geral.columns}\n')
    base_sem_ponderar.index = tabela_geral.columns
    tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

    tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
    print(f'TABELA GERAL:\n{tabela_geral.iloc[:, 0:10]}\n')

Tabela Ponderada Freq Rel: 
MERC_PME                   Mercado PME
Q2                                    
1 a 4 funcionários            0.202335
De 5 a 9                      0.089075
De 10 a 19                    0.334013
De 20 a 49                    0.170153
De 50 a 99 funcionários       0.023767
De 100 a 249 funcionários     0.162343
De 250 a 499 funcionários     0.018314
De 500 a 999 funcionários     0.000000
Acima de 999 funcionários     0.000000

Tabela Sem Ponderação: 
MERC_PME                   Mercado PME
Q2                                    
1 a 4 funcionários                 695
De 5 a 9                           286
De 10 a 19                         635
De 20 a 49                         330
De 50 a 99 funcionários             61
De 100 a 249 funcionários          666
De 250 a 499 funcionários           81
De 500 a 999 funcionários            0
Acima de 999 funcionários            0

Tabela Ponderada Freq Rel: 
MERC_500                   Mercado 500+
Q2                  

In [16]:
Cabecalho = Cabecalho.split(sep=',')
print(f'Cabeçalho:\n{Cabecalho}')
header_above = []
print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
for col in tabela_geral.columns:
    valor = col.split(sep=' - ')[0]
    print(f'valor: {valor}')
    header_above.append(valor)
print(f'\nheader_above:\n{header_above}')

col_series = []
for i, valor in enumerate(Cabecalho):
    # col_names = df[Colunas[i]][pd.notna(df[Colunas[i]])].unique()
    col_names = dict_ord_labels[Colunas[i]]
    print(f'\ncol_names:\n{col_names}')
    for col in col_names:
        col_series.append((Titulo, valor, col))

header = [(Titulo, '', 'GERAL')]
print(f'\ncol_series:\n{col_series}')
header = header + col_series
print(f'\nheader:\n{header}')
print(f'\ntamanho header:\t{len(header)}')
    
header = pd.MultiIndex.from_tuples(header)
print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
print(f'\ntamanho tabela_geral.columns:\t{len(tabela_geral.columns)}')
tabela_geral.columns = header
tabela_geral
print(f'\n\n #===== TABELA GERAL FINAL =====#\n{tabela_geral}\n')
print(f'#===================================================================================#\n')

Cabeçalho:
['Mercado PME', ' Mercado 500+', ' TIM', ' TIM PME', ' TIM 500+', ' TIM Prime', ' VIVO', ' VIVO PME', ' VIVO 500+', ' CLARO', ' CLARO PME', ' CLARO 500+']

tabela_geral.columns:
Index(['Geral', 'Mercado PME', 'Mercado 500+', 'TIM', 'TIM PME', 'TIM 500+',
       'TIM Prime', 'VIVO TOTAL', 'VIVO PME', 'VIVO 500+', 'CLARO TOTAL',
       'CLARO PME', 'CLARO 500+'],
      dtype='object')
valor: Geral
valor: Mercado PME
valor: Mercado 500+
valor: TIM
valor: TIM PME
valor: TIM 500+
valor: TIM Prime
valor: VIVO TOTAL
valor: VIVO PME
valor: VIVO 500+
valor: CLARO TOTAL
valor: CLARO PME
valor: CLARO 500+

header_above:
['Geral', 'Mercado PME', 'Mercado 500+', 'TIM', 'TIM PME', 'TIM 500+', 'TIM Prime', 'VIVO TOTAL', 'VIVO PME', 'VIVO 500+', 'CLARO TOTAL', 'CLARO PME', 'CLARO 500+']

col_names:
['Mercado PME']

col_names:
['Mercado 500+']

col_names:
['TIM']

col_names:
['TIM PME']

col_names:
['TIM 500+']

col_names:
['TIM Prime']

col_names:
['VIVO TOTAL']

col_names:
['VIVO PME']

co

In [17]:
display(tabela_geral)

2. COTA - Qual o número de funcionários registrados da empresa?  (Espontânea - RU)  \
                                                                                                               
                                                                                                       GERAL   
Q2                                                                                                             
1 a 4 funcionários                                                  0.165760                                   
De 5 a 9                                                            0.072974                                   
De 10 a 19                                                          0.273636                                   
De 20 a 49                                                          0.139395                                   
De 50 a 99 funcionários                                             0.019471                                   
De 100 a 249 funcionários                                           0.132997                                   
De 250 a 499 funcionários                                           0.015003                                   
De 500 a 999 funcionários                                           0.107828                                   
Acima de 999 funcionários                                           0.072936                                   
Base Ponderada                                                   3043.999992                                   
Base Sem Ponderar                                                3044.000000                                   

                                                                   \
                           Mercado PME  Mercado 500+          TIM   
                           Mercado PME  Mercado 500+          TIM   
Q2                                                                  
1 a 4 funcionários            0.202335      0.000000     0.223722   
De 5 a 9                      0.089075      0.000000     0.040581   
De 10 a 19                    0.334013      0.000000     0.223093   
De 20 a 49                    0.170153      0.000000     0.115194   
De 50 a 99 funcionários       0.023767      0.000000     0.036390   
De 100 a 249 funcionários     0.162343      0.000000     0.140765   
De 250 a 499 funcionários     0.018314      0.000000     0.012314   
De 500 a 999 funcionários     0.000000      0.596513     0.150148   
Acima de 999 funcionários     0.000000      0.403487     0.057793   
Base Ponderada             2493.752507    550.247485   659.846682   
Base Sem Ponderar          2754.000000    290.000000  1157.000000   

                                                                             \
                               TIM PME    TIM 500+   TIM Prime         VIVO   
                               TIM PME    TIM 500+   TIM Prime   VIVO TOTAL   
Q2                                                                            
1 a 4 funcionários            0.282456    0.000000    0.132829     0.152815   
De 5 a 9                      0.051235    0.000000    0.072905     0.088297   
De 10 a 19                    0.281662    0.000000    0.133564     0.303608   
De 20 a 49                    0.145436    0.000000    0.356729     0.139105   
De 50 a 99 funcionários       0.045944    0.000000    0.216620     0.003269   
De 100 a 249 funcionários     0.177721    0.000000    0.087354     0.132145   
De 250 a 499 funcionários     0.015546    0.000000    0.000000     0.015478   
De 500 a 999 funcionários     0.000000    0.722068    0.000000     0.091401   
Acima de 999 funcionários     0.000000    0.277932    0.000000     0.073882   
Base Ponderada              522.637566  137.209116   79.103095  1578.965123   
Base Sem Ponderar          1047.000000  110.000000  117.000000   992.000000   

                                                                            \
                              VIVO PME   VIVO

In [ ]:
# FUNÇÃO PARA PROCESSAR UMA ÚNICA TABELA
def processar_tabela(bd_dados: pd.DataFrame, lista_labels: pd.DataFrame, 
                     TipoTabela: str, Var_linha: str, Colunas: list, Cabecalho: str, NS_NR: str, Var_ID: str, Var_Pond: str, Titulo: str,
                     valores_BTB: str = '1, 2, 3', valores_TTB: str = '8, 9, 10', Valores_Agrup: str = 'Q8_1, Q8_2, Q8_3', Fecha_100: str = 'SIM'):
    
    df = bd_dados.copy()
    dict_ord_labels = {}

    for col in Colunas:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada no DataFrame.")        
        else:
            df[col], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=col)
            dict_ord_labels[col] = ord_labels
            print(f"\nColuna ordenada (unique): {df[col].unique()}")

    # Etapas de ETL para as colunas principais a serem utilizadas
    # Transformação na variável para a linha da tabela
    if TipoTabela == 'SIMPLES':
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
        else:
            df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=df[Var_linha][pd.notna(df[Var_linha])].unique(), ordered=True)
        
    elif (TipoTabela == 'NPS') | (TipoTabela == 'IPA_10'):
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha] = df[Var_linha].replace('ns/nr', np.nan)
            df[Var_linha] = df[Var_linha].replace('90', np.nan)
            df[Var_linha] = df[Var_linha].replace('99', np.nan)
            df[Var_linha] = df[Var_linha].replace('999', np.nan)
            df[Var_linha] = df[Var_linha].replace('9999', np.nan)
            df[Var_linha] = df[Var_linha].replace(90, np.nan)
            df[Var_linha] = df[Var_linha].replace(99, np.nan)
            df[Var_linha] = df[Var_linha].replace(999, np.nan)
            df[Var_linha] = df[Var_linha].replace(9999, np.nan)
            df[Var_linha] = pd.to_numeric(df[Var_linha], errors='coerce', downcast='integer')
            # df[Var_linha], ord_labels = ordenar_labels(df=df, lista_labels=lista_labels, Variavel=Var_linha)
            df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

            if TipoTabela == 'NPS':
                df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

            elif TipoTabela == 'IPA_10':
                valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

        else:
            # df[Var_linha] = df[Var_linha].replace('90', np.nan)
            # df[Var_linha] = df[Var_linha].replace('99', np.nan)
            # df[Var_linha] = df[Var_linha].replace('999', np.nan)
            # df[Var_linha] = df[Var_linha].replace('9999', np.nan)
            # df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            df[Var_linha] = pd.Categorical(df[Var_linha], categories=ordenar_valores(df[Var_linha]), ordered=True)

            if TipoTabela == 'NPS':
                df['var_agrupada'] = df[Var_linha].apply(classificar_nps)

            elif TipoTabela == 'IPA_10':
                valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
                valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
                df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)

    elif TipoTabela == 'IPA_5':
        if NS_NR == 'NAO':
            df[Var_linha] = df[Var_linha].replace('NS/NR', np.nan)
            df[Var_linha] = df[Var_linha].replace('90', np.nan)
            df[Var_linha] = df[Var_linha].replace('99', np.nan)
            df[Var_linha] = df[Var_linha].replace('999', np.nan)
            df[Var_linha] = df[Var_linha].replace(90, np.nan)
            df[Var_linha] = df[Var_linha].replace(99, np.nan)
            df[Var_linha] = df[Var_linha].replace(999, np.nan)
            valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
            df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
            df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)
        else:
            valores_BTB = [int(v) for v in valores_BTB.split(sep=',')]
            valores_TTB = [int(v) for v in valores_TTB.split(sep=',')]
            df['var_agrupada'] = funcao_agrupamento(df[Var_linha], valores_BTB, valores_TTB)
            df['var_agrupada'] = pd.Categorical(df['var_agrupada'], categories=['BTB', 'Neutro', 'TTB'], ordered=True)
            df[Var_linha], ord_labels = ordenar_labels(df=data, lista_labels=lista_labels, Variavel=Var_linha)
            # df[Var_linha] = pd.Categorical(df[Var_linha], categories=Valores_Agrup, ordered=True)

    elif TipoTabela == 'MULTIPLA':
        if NS_NR == 'NAO':
            df_NS_NR = df.copy()
            
            Valores_Agrup = Valores_Agrup.split(sep=',')
            # Converte as colunas de motivos para string, preservando NaN
            # for c in Valores_Agrup:
            #     df[c] = df[c].astype("object").where(df[c].isna(), df[c].astype(str))
            #     df_NS_NR[c] = df_NS_NR[c].astype("object").where(df_NS_NR[c].isna(), df_NS_NR[c].astype(str))

            bd_motivo = pd.melt(df, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('90', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('99', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('999', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('9999', np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(90, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(99, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(999, np.nan)
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace(9999, np.nan)
            bd_motivo = bd_motivo.dropna(subset=[Var_linha])
            print(f'\nbd_motivo em formato de código:\n{bd_motivo}')
            bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
            bd_motivo[Var_linha] = bd_motivo[Var_linha].replace('NS/NR', np.nan)
            # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)  

            df_limpo = bd_motivo.dropna(subset=[Var_linha])
            print(f'bd_motivo finalizado:\n{df_limpo}')
            df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')
                    
            # Bancos para realizar o cálculo do Índice de Multiplicidade (incluir NS/NR)
            bd_motivo_NS_NR = pd.melt(df_NS_NR, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo_NS_NR = bd_motivo_NS_NR.dropna(subset=[Var_linha])
            bd_motivo_NS_NR = ordenar_labels_multipla(df=bd_motivo_NS_NR, lista_labels=lista_labels, Variavel=Var_linha, 
                                                    Var_Valores_Agrup=Valores_Agrup[0])
            # bd_motivo_NS_NR[Var_linha] = pd.Categorical(bd_motivo_NS_NR[Var_linha], 
            #                                     categories=ordenar_valores(bd_motivo_NS_NR[Var_linha]), ordered=True)
            
            df_NS_NR_limpo = bd_motivo_NS_NR.dropna(subset=[Var_linha])
            print(f'bd_motivo_NS_NR finalizado:\n{df_limpo}')
            df_NS_NR_unico = df_NS_NR_limpo.drop_duplicates(subset=Var_ID, keep='first')    
            
        else:
            Valores_Agrup = Valores_Agrup.split(sep=',')
            bd_motivo = pd.melt(df, 
                        id_vars=Colunas + [Var_Pond] + [Var_ID],
                        value_vars=Valores_Agrup, 
                        var_name='Valores', 
                        value_name=Var_linha)
            bd_motivo = bd_motivo.dropna(subset=[Var_linha])
            bd_motivo = ordenar_labels_multipla(df=bd_motivo, lista_labels=lista_labels, Variavel=Var_linha, Var_Valores_Agrup=Valores_Agrup[0])
            # bd_motivo[Var_linha] = pd.Categorical(bd_motivo[Var_linha], 
            #                                     categories=ordenar_valores(bd_motivo[Var_linha]), ordered=True)
            
            df_limpo = bd_motivo.dropna(subset=[Var_linha])
            df_unico = df_limpo.drop_duplicates(subset=Var_ID, keep='first')


    #======= Criação da Tabela Geral =======#
    tabela_geral = []
    tabelas_pond = []
    tabelas_pond_freq_abs = []
    tabelas_sem_pond = []
    aux_tabelas_pond = []
    aux_tabelas_sem_pond = []

    tbl_pond_freq_abs_respondentes = []
    tbl_sem_pond_respondentes = []

    tbl_pond_freq_abs_respostas_NS_NR = []
    tbl_pond_freq_abs_respondentes_NS_NR = []

    if TipoTabela == 'MULTIPLA':
        if NS_NR == 'NAO':
            banco = df_NS_NR_unico
        else:
            banco = df_unico

        if NS_NR == 'NAO':
            banco_pivotado = bd_motivo_NS_NR
        else:
            banco_pivotado = bd_motivo

        i = 0    
        for col in Colunas:

            # Gerar Tabelas Ponderadas de Frequência Absoluta com o banco empilhado
            tabela = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            if Fecha_100 == 'SIM':
                tabela = tabela.div(tabela.sum())
                tabelas_pond.append(tabela)
            else:
                tabelas_pond.append(tabela)

            tabela = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respostas_NS_NR.append(tabela)

            tabela = pd.pivot_table(df_unico, values=Var_Pond, index=Var_linha, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respondentes.append(tabela)

            tabela = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, columns=col, aggfunc='sum')
            tbl_pond_freq_abs_respondentes_NS_NR.append(tabela)

            tabela = pd.crosstab(df_unico[Var_linha], df_unico[col], dropna=False)
            tabela = tabela.fillna(0)
            if len(tabela) == 0:
                tabela = pd.DataFrame(0, index=df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique(), 
                                    columns=df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique())
                print(f'{df_unico[Var_linha][pd.notna(df_unico[Var_linha])].unique()}\n')
                print(f'{df_unico[Colunas[i-1]][pd.notna(df_unico[Colunas[i-1]])].unique()}\n')
            print(f'tabela:\n{tabela}')
            tbl_sem_pond_respondentes.append(tabela)
            i += 1

        tabela_geral = pd.concat(tabelas_pond, axis=1)
        tbl_pond_freq_abs_respondentes = pd.concat(tbl_pond_freq_abs_respondentes, axis=1)
        tbl_sem_pond_respondentes = pd.concat(tbl_sem_pond_respondentes, axis=1)
        tbl_pond_freq_abs_respostas_NS_NR = pd.concat(tbl_pond_freq_abs_respostas_NS_NR, axis=1)
        tbl_pond_freq_abs_respondentes_NS_NR = pd.concat(tbl_pond_freq_abs_respondentes_NS_NR, axis=1)
        
        if Fecha_100 != 'SIM':
            soma_base_ponderada = tbl_pond_freq_abs_respondentes.sum()
            tabela_geral = tabela_geral.div(soma_base_ponderada)
        
        print(f'{tabela_geral}\n')


        # Trazendo a coluna de valores gerais
        soma_colunas = tbl_pond_freq_abs_respondentes.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        valores_gerais = pd.pivot_table(bd_motivo, values=Var_Pond, index=Var_linha, aggfunc='sum')
        if Fecha_100 == 'SIM':
            percentual_geral = valores_gerais.div(valores_gerais.sum()).sort_index()
            print(f'\npercentual_geral:\n{percentual_geral}')
            print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')
        else:
            percentual_geral = valores_gerais.div(base_ponderada[0]).sort_index()
            print(f'\npercentual_geral:\n{percentual_geral}')
            print(f'\nsoma de percentual_geral:\t{percentual_geral.sum()}')

        # tabela_geral = tabela_geral.div(base_ponderada[1:])
        tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
        print(f'{tabela_geral}\n')

        # Valores para Base Ponderada
        soma_colunas = tbl_pond_freq_abs_respondentes.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df_unico, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        tabela_geral.loc['Base Ponderada'] = base_ponderada

        # Valores para Base Sem Ponderar
        valores_gerais_respondentes = df_unico[Var_ID].value_counts()
        soma_colunas = pd.Series(tbl_sem_pond_respondentes.sum())
        print(f'\n{soma_colunas}\n')
        base_sem_ponderar = pd.Series(valores_gerais_respondentes.sum())
        base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
        print(f'base sem ponderar:\n{base_sem_ponderar.index}\n')
        print(f'{len(base_sem_ponderar.index)}\n')
        print(f'tabela geral:\n{tabela_geral.columns}\n')
        print(f'{len(tabela_geral.columns)}\n')
        base_sem_ponderar.index = tabela_geral.columns
        tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

        # Valores para Base Ponderada - Respostas
        soma_colunas_respostas = tbl_pond_freq_abs_respostas_NS_NR.sum()
        soma_colunas_respostas = pd.Series(soma_colunas_respostas)
        base_ponderada_respostas = pd.pivot_table(banco_pivotado, values=Var_Pond, index=Var_linha, aggfunc='sum')
        base_ponderada_respostas = pd.Series(base_ponderada_respostas.sum())
        base_ponderada_respostas = pd.concat([base_ponderada_respostas, soma_colunas_respostas])

        # Valores para Base Ponderada - Respondentes
        soma_colunas = tbl_pond_freq_abs_respondentes_NS_NR.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(banco, values=Var_Pond, index=Var_ID, aggfunc='sum')
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])

        # Índice de Multiplicidade (Total de respostas / Total de respondentes)
        indice_multiplicidade = base_ponderada_respostas / base_ponderada
        tabela_geral.loc['Índice de Multiplicidade'] = indice_multiplicidade

        tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
        print(f'{tabela_geral}\n')

    else:
        if df[Var_linha].isna().all():
            # df[Var_linha] = df[Var_linha].astype('float').fillna(0)
            df[Var_linha] = df[Var_linha].cat.add_categories(['Não possui categoria']).fillna('Não possui categoria')
            print(f'Variável em branco\n{df[Var_linha]}\n')

        for col in Colunas:
            # Gerar Tabelas Ponderadas de frequência absoluta
            tabela = pd.crosstab(df[Var_linha], df[col], values=df[Var_Pond], aggfunc='sum')
            tabelas_pond_freq_abs.append(tabela)

            # Gerar Tabelas Ponderadas de frequência relativa 
            tabela = tabela.div(tabela.sum())
            tabela = tabela.fillna(0)
            tabelas_pond.append(tabela)
            print(f'Tabela Ponderada Freq Rel: \n{tabela}\n')

            # Gerar Tabelas Sem Ponderação
            tabela = pd.crosstab(df[Var_linha], df[col], dropna=False)
            tabela = tabela.fillna(0)
            print(f'Tabela Sem Ponderação: \n{tabela}\n')
            if len(tabela) == 0:
                tabela = pd.DataFrame(0, index=df[Var_linha][pd.notna(df[Var_linha])].unique(), 
                                    columns=df[col][pd.notna(df[col])].unique())
                print(f'Tabela Sem Ponderação: \n{tabela}\n')
            tabelas_sem_pond.append(tabela)

            # Gerar Tabelas para valores agrupados
            if 'var_agrupada' in df.columns:
                tabela = pd.crosstab(df['var_agrupada'], df[col], values=df[Var_Pond], aggfunc='sum')
                tabela = tabela.div(tabela.sum())
                print(f'Tabela Valores Agrupados: \n{tabela}\n')
                aux_tabelas_pond.append(tabela)

                # Tabelas sem ponderação
                tabela = pd.crosstab(df['var_agrupada'], df[col], dropna=False)
                print(f'Tabela Valores Agrupados Sem Ponderação: \n{tabela}\n')
                aux_tabelas_sem_pond.append(tabela)

        tabela_geral = pd.concat(tabelas_pond, axis=1)
        tabelas_pond_freq_abs = pd.concat(tabelas_pond_freq_abs, axis=1)
        tabelas_sem_pond = pd.concat(tabelas_sem_pond, axis=1)

        # Trazendo a coluna de valores gerais
        valores_gerais_pond = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
        print(f'\nValores GERAL:\n{valores_gerais_pond}')
        percentual_geral = valores_gerais_pond.div(valores_gerais_pond.sum()).sort_index()
        print(f'\PERCENTUAL GERAL:\n{percentual_geral}')

        if 'var_agrupada' in df.columns:
            aux_tabelas_pond = pd.concat(aux_tabelas_pond, axis=1)
            aux_tabelas_sem_pond = pd.concat(aux_tabelas_sem_pond, axis=1)

            # Percentual para os valores gerais da variável agrupada
            valores_gerais_aux = pd.pivot_table(df, values=Var_Pond, index='var_agrupada', aggfunc='sum')
            percentual_geral_aux = valores_gerais_aux.div(valores_gerais_aux.sum()).sort_index()
            tabela_geral = pd.concat([tabela_geral, aux_tabelas_pond], axis=0)
            percentual_geral = pd.concat([percentual_geral, percentual_geral_aux], axis=0)
            tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)
        else:
            tabela_geral = pd.concat([percentual_geral, tabela_geral], axis=1)

        # Multiplicando cada linha pelo seu índice+1
        if TipoTabela == 'NPS':
            # Gerar o NPS
            nps = tabela_geral.loc['Promotor'] - tabela_geral.loc['Detrator']
            tabela_geral.loc['NPS'] = nps
            # Gerar as médias
            media_tabela = tabelas_pond_freq_abs.copy()
            for i in range(len(tabelas_pond_freq_abs)):
                media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
            media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

            # Cálculo da média geral
            df_limpo = df.dropna(subset=[Var_linha])
            media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
            media_geral = pd.Series(media_geral)
            media_tabela = pd.concat([media_geral, media_tabela], axis=0)
            media_tabela.index = tabela_geral.columns
            tabela_geral.loc['Media'] = media_tabela

        elif (TipoTabela == 'SATISFACAO') | (TipoTabela == 'IPA_10') :
            media_tabela = tabelas_pond_freq_abs.copy()
            for i in range(len(tabelas_pond_freq_abs)):
                media_tabela.iloc[i] = tabelas_pond_freq_abs.iloc[i] * (i + 1)
            media_tabela = media_tabela.sum().div(tabelas_pond_freq_abs.sum())

            # Cálculo da média geral
            df_limpo = df.dropna(subset=[Var_linha])
            media_geral = sum(np.array(df_limpo[Var_linha]) * np.array(df_limpo[Var_Pond])) / sum(np.array(df_limpo[Var_Pond]))
            media_geral = pd.Series(media_geral)
            media_tabela = pd.concat([media_geral, media_tabela], axis=0)

            media_tabela.index = tabela_geral.columns
            tabela_geral.loc['Media'] = media_tabela

        else:
            tabela_geral

        tabela_geral = tabela_geral.fillna(0)

        # Valores para Base Ponderada
        tabelas_pond_freq_abs = tabelas_pond_freq_abs.fillna(0)
        soma_colunas = tabelas_pond_freq_abs.sum()
        soma_colunas = pd.Series(soma_colunas)
        base_ponderada = pd.pivot_table(df, values=Var_Pond, index=Var_linha, aggfunc='sum', observed=False)
        base_ponderada = pd.Series(base_ponderada.sum())
        base_ponderada = pd.concat([base_ponderada, soma_colunas])
        # base_ponderada.index = tabela_geral.columns
        tabela_geral.loc['Base Ponderada'] = base_ponderada

        # Valores para Base Sem Ponderar
        tabelas_sem_pond = tabelas_sem_pond.fillna(0)
        soma_colunas = pd.Series(tabelas_sem_pond.sum())
        print(f'{soma_colunas}\n')
        valores_gerais = df[Var_linha].value_counts().sort_index()
        base_sem_ponderar = pd.Series(valores_gerais.sum())
        base_sem_ponderar = pd.concat([base_sem_ponderar, soma_colunas])
        print(f'{base_sem_ponderar.index}\n')
        print(f'{tabela_geral.columns}\n')
        base_sem_ponderar.index = tabela_geral.columns
        tabela_geral.loc['Base Sem Ponderar'] = base_sem_ponderar

        tabela_geral.rename(columns={tabela_geral.columns[0]: 'Geral'}, inplace=True)
        print(f'TABELA GERAL:\n{tabela_geral.iloc[:, 0:10]}\n')


    Cabecalho = Cabecalho.split(sep=',')
    print(f'Cabeçalho:\n{Cabecalho}')
    header_above = []
    print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
    for col in tabela_geral.columns:
        valor = col.split(sep=' - ')[0]
        print(f'valor: {valor}')
        header_above.append(valor)
    print(f'\nheader_above:\n{header_above}')

    col_series = []
    for i, valor in enumerate(Cabecalho):
        # col_names = df[Colunas[i]][pd.notna(df[Colunas[i]])].unique()
        col_names = dict_ord_labels[Colunas[i]]
        print(f'\ncol_names:\n{col_names}')
        for col in col_names:
            col_series.append((Titulo, valor, col))

    header = [(Titulo, '', 'GERAL')]
    print(f'\ncol_series:\n{col_series}')
    header = header + col_series
    print(f'\nheader:\n{header}')
    print(f'\ntamanho header:\t{len(header)}')
        
    header = pd.MultiIndex.from_tuples(header)
    print(f'\ntabela_geral.columns:\n{tabela_geral.columns}')
    print(f'\ntamanho tabela_geral.columns:\t{len(tabela_geral.columns)}')
    tabela_geral.columns = header
    
    return tabela_geral

In [5]:
table = processar_tabela(bd_dados=df, lista_labels=lista_labels, TipoTabela=TipoTabela, Var_linha=Var_linha, Colunas=Colunas, Cabecalho=Cabecalho,
                         NS_NR=NS_NR, valores_BTB=valores_BTB, valores_TTB=valores_TTB, Valores_Agrup=Valores_Agrup, Fecha_100=Fecha_100,
                         Var_ID=Var_ID, Var_Pond=Var_Pond, Titulo=Titulo)
table

ValueError: Coluna 'M' não encontrada no DataFrame.

In [6]:
def somar(valor1: int = 2, valor2: int = 3):
    return valor1 + valor2

print(somar())

5
